# MarshPulse · V3 / rigorous Kaggle GPU challenger tournament

**Real training, not a fabricated demonstration.** This notebook collects ESA Sentinel-2 L2A imagery, creates two training areas and an **entirely different spatial test area**, trains a persistence-residual ConvGRU challenger across three fixed seeds, and compares it to persistence and 12-month seasonality on *exactly the same clear pixels*. The algorithm may reject all neural candidates and keep persistence as champion.

**Non-negotiable:** labels are MNDWI/NDVI satellite-derived proxies, **not independently surveyed water truth**. No field deployment, water-allocation decisions, or "ChatGPT-level reliability" is permitted by this notebook. All source IDs and dates are logged. This is a stage in scientific validation, not an operational authorization.

**Before Run All:** Settings → Accelerator → **GPU**; turn **Internet on** (if your Kaggle account allows it). Kaggle access and GPU quotas depend on your account; a connected GitHub account does not automatically grant Kaggle execution permissions.

In [ ]:
# Dependencies (CPU torch is preinstalled on many Kaggle images; verify CUDA below).
%pip -q install "numpy>=1.26,<3" "rasterio>=1.3,<2" "pystac-client>=0.8,<1" "planetary-computer>=1.0,<2"
import os,sys,subprocess,json,hashlib,platform,zipfile
from pathlib import Path
import numpy as np
import torch
print("Python",sys.version.split()[0],"torch",torch.__version__,"CUDA",torch.cuda.is_available())
print("GPU",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE")
assert torch.cuda.is_available(), "Set Kaggle accelerator to GPU before a real V3 tournament (no silent CPU fallback)."

## 1 · Source code + version lock

The notebook clones the public GitHub repository and stores its exact Git commit with the results. The worktree must be a real checked-out repository; no pasted mock training code.

In [ ]:
ROOT=Path("/kaggle/working/MarshPulse")
if not (ROOT/"ml/train_v3.py").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/ahpa5246-lgtm/MarshPulse.git",str(ROOT)],check=True)
commit=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
assert len(commit)==40 and (ROOT/"ml/marshnet_v3.py").is_file()
print("SOURCE COMMIT:",commit)
OUTPUT=Path("/kaggle/working/marshpulse_v3_evidence")
OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/"source_commit.txt").write_text(commit+"\n")

## 2 · Predeclared, geographically disjoint study areas

These are **small pilot bounding rectangles centered near official marsh sites, not wetland boundaries**. Geographical centers follow [UNESCO's *Ahwar of Southern Iraq* geographic reference](https://whc.unesco.org/document/169030), not from a guessed synthetic grid. Training: Hawizeh and Central Marshes. **Spatial holdout: Hammar**, excluded entirely from training, normalization, checkpoint selection and hyperparameter search.

Fixed observation period October 2022 – September 2026 (48 complete months). A single AOI cannot establish geographic generalization. Increasing pixels or duration will raise acquisition cost. If any region lacks sufficient clear real scenes, the notebook fails instead of fabricating data.

In [ ]:
AOIS={
 "hawizeh_train":[47.64,31.53,47.74,31.63],
 "central_train":[47.00,31.04,47.10,31.14],
 "hammar_spatial_test":[46.66,30.79,46.76,30.89],
}
START,END="2022-10","2026-09"
GRID_PIXELS=64
MAX_SCENES_PER_MONTH=2
MIN_COVERAGE=.20
SAT_ROOT=OUTPUT/"source_imagery"
paths={}
for name,bbox in AOIS.items():
    path=SAT_ROOT/name
    path.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,str(ROOT/"ml/collect_sentinel.py"),
        "--bbox",*map(str,bbox),"--start",START,"--end",END,
        "--pixels",str(GRID_PIXELS),"--max-scenes",str(MAX_SCENES_PER_MONTH),
        "--min-coverage",str(MIN_COVERAGE),"--out",str(path)]
    print("ACQUIRING",name,"AOI",bbox,flush=True)
    subprocess.run(cmd,check=True)
    paths[name]=path/"hawizeh_s2_monthly.npz"
print("Real acquired Sentinel cubes:",paths)

## 3 · Audit input integrity **before** model training

Every monthly frame must have an acquisition record. Missing/cloudy pixels are not relabeled as dry. The water proxy may miss vegetated water and must not be treated as field truth. Threshold-sensitive findings are exposed, never optimized against the final test labels.

In [ ]:
for name,npz in paths.items():
    output=OUTPUT/("quality_"+name+".json")
    subprocess.run([sys.executable,str(ROOT/"ml/quality_gate.py"),
         "--input",str(npz),"--out",str(output)],check=True)
    summary=json.loads(output.read_text())
    manifest=json.loads(npz.with_name("satellite_manifest.json").read_text())
    assert len(manifest["months"])==48
    assert sum(m["fraction_valid"]>=MIN_COVERAGE for m in manifest["months"])>=max(12,int(48*.55))
    assert all(m["scenes"] for m in manifest["months"] if m["fraction_valid"]>0)
    # A completely cloudy month is represented by VALID=0, not fabricated as dry.
    print(name, "months",len(summary["months"]),
          "months_with_wet_proxy",summary["months_with_wet_proxy"],
          "wet_fraction",summary["overall_wet_fraction"],
          "warnings",summary["warnings"])
    assert summary["total_observed_pixels"]>0, "No observed satellite pixels"

## 4 · Train and select a challenger without looking at test targets

Run a three-seed, proper-loss ConvGRU challenger. Each region uses chronological training and validation months. Checkpoint selection requires **validation Brier beating previous-month persistence** on the same observed pixels and enough validation wet examples. No model is promoted merely because it achieved a low training loss. The Hammar AOI is untouched until the final, post-selection report.

In [ ]:
RESULT=OUTPUT/"evaluation"
cmd=[sys.executable,str(ROOT/"ml/train_v3.py"),
 "--train-input",str(paths["hawizeh_train"]),
 "--train-input",str(paths["central_train"]),
 "--spatial-test-input",str(paths["hammar_spatial_test"]),
 "--out",str(RESULT),
 "--epochs","24","--patience","6","--batch","2","--base","16",
 "--lookback","4","--seeds","17,42,2026","--min-wet-valid","25"]
print("Running GPU training (no synthetic input)",flush=True)
subprocess.run(cmd,check=True)
report=json.loads((RESULT/"v3_evaluation.json").read_text())
print("CHAMPION",report["selected_champion"]["type"])
print("training_device",report["device"])
for v in report["selected_champion"]["validation_challengers"]:
    print("seed",v["seed"],"best_epoch",v["best_epoch"],
          "validation_Brier",v["val_brier"],
          "persistence_validation_Brier",v["val_persistence_brier"],
          "wet_F1",v["validation_candidate_wet_f1"],
          "best_baseline_wet_F1",v["validation_best_simple_wet_f1"],
          "passed",v["passed_validation_gate"])

## 5 · Final sealed assessment, independent geographical location

The test set is read only after validation-only selection. We retain **all negative results**. If none beats the baseline, the machine reports PERSISTENCE_BASELINE—not an impressive-sounding but failed deep network. This notebook cannot certify groundwater, river discharge, restoration interventions, field observations or reliable drought warnings.

In [ ]:
report=json.loads((RESULT/"v3_evaluation.json").read_text())
assert not report["operationally_reliable"] and not report["independent_field_labels"]
assert report["spatial_transfer_evaluated"]
for area in report["test_evaluations"]:
    print("\\nAREA:",area["region"],"shared_fraction",area["shared_observed_fraction"])
    print("ACTUAL SELECTION:",report["selected_champion"]["type"],
          "METRIC:",area["selected_champion_metric"],
          "PROMOTED?",area["candidate_was_promoted"])
    print("MONTHS WITH SHARED OBSERVATIONS:",area["months_with_shared_observations"])
    for m in area["monthly_paired_scores"]:
        print(" ",m["month"],m["status"],"shared",m["shared_pixels"])
    for method,metrics in area["metrics"].items():
        print(f'  {method:22s} Brier={metrics["brier"]:.6f} F1={metrics["f1"]} '
              f'wet={metrics["wet_truth_pixels"]} pixels={metrics["evaluated_pixels"]}')
print("\\nOPERATIONAL RELIABILITY: NOT DEMONSTRATED")
(OUTPUT/"experiment_summary.json").write_text(json.dumps({
 "source_commit":commit,"train_bboxes":AOIS,
 "window":[START,END],"device":report["device"],
 "model_selection":report["selected_champion"]["type"],
 "field_validated":False,"spatial_test":True
},indent=2))

## 6 · Download all evidence

Archive study-area manifests, source rasters, monthly quality audits, trained challenger (if selected), complete seed-by-seed validation history, and test results. This may be several MB. Inspect the output files via the right-hand Kaggle sidebar.

**Scientific next gate:** independently annotated Sentinel/high-resolution imagery and leave-one-wetland-out evaluation across seasons/years. JRC GSW 1.4 monthly history ends in **2021**, so it is not an independent 2022–2026 validation source. Neither the Kaggle free GPU nor three random seeds can replace independent reference labels.

In [ ]:
evidence=OUTPUT/"MarshPulse_V3_Kaggle_Real_Research.zip"
with zipfile.ZipFile(evidence,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=3) as archive:
    for item in OUTPUT.rglob("*"):
        if item.is_file() and item!=evidence:
            archive.write(item,item.relative_to(OUTPUT))
print("RESEARCH ARCHIVE:",evidence)
print("BYTES:",evidence.stat().st_size)
print("SHA256:",hashlib.sha256(evidence.read_bytes()).hexdigest())
print("IMPORTANT: No field validation, certification, or guaranteed improvement.")